
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Clasificación de tweets con distintas representaciones: TF, TF-IDF y Word Embeddings

# Introducción

El objetivo de este notebook es comparar distintas formas de representar texto como *features* para un modelo de clasificación, usando como caso el dataset **HatEval** (SemEval-2019 Task 5), que ya vienen utilizando en las prácticas de este taller. El dataset viene dividido en tres splits, cada uno en su propio archivo dentro de `data/`:

- `hateval_train_df.csv`: split de **entrenamiento**. Lo usamos para ajustar (`fit`) cada modelo.
- `hateval_dev_df.csv`: split de **validación**. Lo usamos para elegir el mejor valor del hiperparámetro de regularización de cada modelo.
- `hateval_test_df.csv`: split de **test**. Lo usamos únicamente al final, para evaluar el desempeño de cada modelo sobre datos que no participaron ni del entrenamiento ni de la elección de hiperparámetros.

Cada fila de estos archivos es un tweet con las siguientes columnas relevantes:

- `id`: identificador del tweet
- `text`: el texto del tweet
- `language`: idioma del tweet (`en` o `es`)
- `HS`: 1 si el tweet contiene discurso de odio (*hate speech*), 0 en caso contrario
- `TR`, `AG`: otras anotaciones (agresividad, si el odio está dirigido a un individuo o a un grupo) que no vamos a usar acá

La tarea de clasificación va a ser predecir `HS` (discurso de odio sí/no) a partir del texto del tweet.

Como en la última sección vamos a usar **embeddings preentrenados en español** (SBWCE), y los tres splits tienen tweets tanto en inglés como en español, nos vamos a quedar solamente con los tweets en español (`language == 'es'`) de cada split para poder comparar las tres representaciones sobre los mismos datos.

Al igual que en el ejemplo con reseñas de Amazon (`cap0/ejemplo_clasificacion.ipynb`), vamos a entrenar en cada caso una regresión logística regularizada por LASSO (penalización L1), variando el hiperparámetro de regularización $C$ (a menor $C$, mayor regularización). A diferencia de ese ejemplo, acá **no** usamos validación cruzada (K-Fold) para elegir $C$: como ya contamos con un split de validación (`dev`) independiente, entrenamos cada candidato sobre `train` y elegimos el que mejor performa sobre `dev`. `test` queda completamente afuera de ese proceso, y solo se usa al final para reportar el desempeño de cada modelo ya elegido.

Vamos a comparar tres formas de vectorizar los tweets:

1. **TF** (*Term Frequency*, bolsa de palabras con conteos)
2. **TF-IDF** (*Term Frequency - Inverse Document Frequency*)
3. **Word embeddings preentrenados** (promedio de vectores de palabras)

Además de las métricas de desempeño (ROC AUC, accuracy, precision, recall, F1), en TF y TF-IDF vamos a mirar los coeficientes ($\beta$) del modelo para interpretar qué palabras aprendió a asociar con discurso de odio. Con embeddings esto no va a ser posible de la misma forma — lo vamos a explicar en esa sección.

In [ ]:
## Ejecutar para descargar los embeddings preentrenados en español (SBWCE)
!wget -P ./models https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz && gunzip ./models/SBW-vectors-300-min5.bin.gz
!pip install gensim
!git clone https://github.com/gefero/factor_data_tuto_NLP_SICSS.git

# TF con LASSO

## Preparación de los datos

Cargamos los tres splits de HatEval (`hateval_train_df.csv`, `hateval_dev_df.csv` y `hateval_test_df.csv`), nos quedamos en cada uno con los tweets en español y preprocesamos el texto:

1. Convertimos a minúsculas
2. Eliminamos URLs y menciones (`@usuario`), que son ruido propio de los tweets
3. Eliminamos signos de puntuación
4. Reemplazamos números por la palabra `DIGITO`
5. Eliminamos acentos y caracteres no ASCII

Para cada split, `X` va a ser el texto preprocesado y `y` la variable binaria `HS` (discurso de odio).

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_path = './factor_data_tuto_NLP_SICSS/data/hateval_train_df.csv'
dev_path = './factor_data_tuto_NLP_SICSS/data/hateval_dev_df.csv'
test_path = './factor_data_tuto_NLP_SICSS/data/hateval_test_df.csv'

In [ ]:
# Función para preprocesar el texto
def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()

    # Eliminar URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Eliminar menciones (@usuario)
    text = re.sub(r'@\w+', ' ', text)

    # Reemplazar puntuación
    text = re.sub(r'[^\w\s]', ' ', text)

    # Reemplazar números por 'DIGITO'
    text = re.sub(r'\d+', 'DIGITO', text)

    # Reemplazar caracteres no ASCII (acentos, etc.)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    # Colapsar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Cargamos un split y nos quedamos con los tweets en español
def load_hateval_split(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    df['text_clean'] = df['text'].apply(preprocess_text)
    return df

train_df = load_hateval_split(train_path)
dev_df = load_hateval_split(dev_path)
test_df = load_hateval_split(test_path)

# Preparamos las variables para el modelo
X_train, y_train = train_df['text_clean'], train_df['HS']
X_dev, y_dev = dev_df['text_clean'], dev_df['HS']
X_test, y_test = test_df['text_clean'], test_df['HS']

print(f"Train: {len(X_train)} tweets | Dev: {len(X_dev)} tweets | Test: {len(X_test)} tweets")

## Cómo se ve una matriz de Term Frequency

Antes de armar el pipeline completo, vale la pena ver qué produce realmente un `CountVectorizer`: una matriz donde cada fila es un documento (tweet) y cada columna una palabra del vocabulario, con la cantidad de veces que esa palabra aparece en ese documento.

Usamos acá solo 4 oraciones cortas de ejemplo (no el corpus completo) y solo unigramas, para que la matriz sea chica y legible. Con miles de tweets y ngramas de 1-2 palabras como en la sección siguiente, esta misma matriz tendría decenas de miles de columnas y sería casi toda ceros (una matriz **dispersa**, *sparse*) — por eso sklearn la guarda en un formato disperso en vez de como un array denso.

In [ ]:
# Oraciones de ejemplo (no son parte del dataset, son solo para ilustrar)
ejemplo_tweets = [
    "no me gustan los inmigrantes en mi barrio",
    "los inmigrantes trabajan mucho en mi barrio",
    "amo la diversidad y a los inmigrantes",
    "odio a la gente que no piensa como yo",
]

# Solo unigramas acá, a diferencia del pipeline real (1-2 gramas), para que
# el vocabulario resultante sea chico y la matriz se pueda leer de un vistazo
vectorizer_ejemplo = CountVectorizer(ngram_range=(1, 1))
matriz_ejemplo = vectorizer_ejemplo.fit_transform(ejemplo_tweets)

# Convertimos la matriz dispersa a un DataFrame denso solo para visualizarla
# (con el corpus real, nunca convertirías la matriz completa a denso: no entraría en memoria)
tf_df = pd.DataFrame(
    matriz_ejemplo.toarray(),
    columns=vectorizer_ejemplo.get_feature_names_out(),
    index=[f'tweet {i+1}' for i in range(len(ejemplo_tweets))]
)
tf_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Rampa secuencial azul (superficie -> azul oscuro) para representar magnitud:
# 0 se funde con el fondo, valores más altos se ven más oscuros/saturados
blue_ramp = mcolors.LinearSegmentedColormap.from_list(
    'blue_seq', ['#fcfcfb', '#cde2fb', '#86b6ef', '#3987e5', '#184f95']
)

fig, ax = plt.subplots(figsize=(9, 3))
fig.patch.set_facecolor('#fcfcfb')

# sns.heatmap ya resuelve lo que antes armábamos a mano: la separación entre
# celdas (linewidths/linecolor) y el contraste del texto (blanco sobre
# celdas oscuras, oscuro sobre claras, según la luminosidad de cada celda)
sns.heatmap(
    tf_df, annot=True, fmt='d', cmap=blue_ramp, vmin=0, cbar=False,
    linewidths=2, linecolor='#fcfcfb', annot_kws={'fontsize': 9}, ax=ax
)
ax.set_facecolor('#fcfcfb')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', color='#52514e', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, color='#52514e', fontsize=9)
ax.set_title('Matriz de Term Frequency (ejemplo ilustrativo, solo unigramas)',
             color='#0b0b0b', fontsize=11)
fig.tight_layout()
plt.show()

## Búsqueda de hiperparámetros usando el split de validación

Armamos un pipeline con:

1. **Vectorización TF**: `CountVectorizer` con ngramas de 1 y 2 palabras
2. **Regresión logística LASSO** (penalización L1, solver `liblinear`)

Para elegir el mejor valor de `C` (inverso de la fuerza de regularización) probamos 30 valores en escala logarítmica entre $10^{-10}$ y $10^{1}$: para cada uno, entrenamos el pipeline sobre `train` y calculamos el ROC AUC sobre `dev`. Nos quedamos con el `C` que da mejor ROC AUC en `dev`.

In [ ]:
# Creamos una grilla de valores para el parámetro C (inverso de la penalización)
C_values = np.logspace(-10, 1, 30)

# Pipeline con TF y LASSO
pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Almacenaremos los resultados de la búsqueda aquí
val_results = []

In [ ]:
%%time
# Para cada valor de C
for C in C_values:
    pipeline_tf.set_params(classifier__C=C)

    # Entrenamos sobre train y evaluamos sobre dev
    pipeline_tf.fit(X_train, y_train)
    dev_proba = pipeline_tf.predict_proba(X_dev)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })

# Convertimos resultados a DataFrame
val_results_df = pd.DataFrame(val_results)

# Encontramos el mejor C
best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_tf = val_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C_tf:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

## Modelo final

Entrenamos el pipeline final sobre `train` con el mejor valor de `C` encontrado y evaluamos sobre `test` (que no participó ni del entrenamiento ni de la elección de hiperparámetros) con ROC AUC, accuracy, precision, recall y F1.

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

final_pipeline_tf.fit(X_train, y_train)
y_pred = final_pipeline_tf.predict(X_test)
y_pred_proba = final_pipeline_tf.predict_proba(X_test)[:, 1]

# Métricas finales
results_tf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

for metric, value in results_tf.items():
    print(f"{metric}: {value:.3f}")

## Interpretación de los coeficientes

Como el clasificador es una regresión logística **lineal** sobre features interpretables (cada columna del vectorizador es una palabra o bigrama), podemos mirar directamente los coeficientes (`coef_`) para ver qué aprendió el modelo — no hace falta ninguna técnica de interpretabilidad externa (SHAP, LIME, etc.).

Dos cosas para tener en cuenta al leerlos:

- **LASSO selecciona variables**: la penalización L1 empuja a *exactamente cero* los coeficientes de la enorme mayoría de las ~64 mil columnas del vocabulario. Solo un puñado de n-gramas "sobrevive" con coeficiente distinto de cero — son los que el modelo realmente usa para predecir.
- **Signo y magnitud**: un coeficiente positivo empuja la predicción hacia `HS=1` (discurso de odio); uno negativo, hacia `HS=0`. Como acá el vectorizador es `CountVectorizer` (conteos crudos), `exp(beta)` tiene una lectura directa como **odds ratio**: "cada aparición adicional de este término multiplica las odds de que el tweet sea odio por `exp(beta)`" (manteniendo todo lo demás constante).

Nota: como el dataset es de discurso de odio, es esperable que los términos con coeficiente más alto incluyan insultos explícitos — es justamente lo que estamos buscando ver.

In [ ]:
# Extraemos el vocabulario del vectorizador y los coeficientes del clasificador
# (mismo orden: la posición i del vocabulario corresponde a coef_[0][i])
feature_names_tf = final_pipeline_tf.named_steps['vectorizer'].get_feature_names_out()
coefs_tf = final_pipeline_tf.named_steps['classifier'].coef_[0]

coef_df_tf = pd.DataFrame({'feature': feature_names_tf, 'coef': coefs_tf})
coef_df_tf['odds_ratio'] = np.exp(coef_df_tf['coef'])

n_total = len(coef_df_tf)
n_nonzero = int((coef_df_tf['coef'] != 0).sum())
print(f"Features totales: {n_total} | con coeficiente != 0 (seleccionadas por LASSO): "
      f"{n_nonzero} ({n_nonzero / n_total * 100:.1f}%)")

top_pos_tf = coef_df_tf.sort_values('coef', ascending=False).head(15)
top_neg_tf = coef_df_tf.sort_values('coef', ascending=True).head(15)

print("\nTop 15 términos que más empujan hacia HS=1 (discurso de odio):")
display(top_pos_tf)

print("\nTop 15 términos que más empujan hacia HS=0 (no odio):")
display(top_neg_tf)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combinamos los 15+15 términos y los ordenamos por coeficiente para el gráfico
top_terms_tf = pd.concat([top_neg_tf, top_pos_tf]).sort_values('coef')

# Codificación diverging: rojo = empuja hacia odio, azul = empuja hacia no-odio
# (los colores de identidad TF/TF-IDF del resto del notebook no aplican acá:
# esto codifica la clase, no la representación)
colors_coef = ['#e34948' if c > 0 else '#2a78d6' for c in top_terms_tf['coef']]

fig, ax = plt.subplots(figsize=(7, 8))
fig.patch.set_facecolor('#fcfcfb')
sns.barplot(data=top_terms_tf, x='coef', y='feature', hue='feature',
            palette=colors_coef, legend=False, ax=ax)
ax.axvline(0, color='#898781', linewidth=1)
ax.set_facecolor('#fcfcfb')
ax.set_xlabel('coeficiente (beta)', color='#52514e')
ax.set_ylabel('')
ax.set_title('Términos más predictivos — TF + LASSO', color='#0b0b0b', fontsize=11)
ax.tick_params(colors='#52514e')
sns.despine(ax=ax, left=True)
fig.tight_layout()
plt.show()

# TF-IDF con LASSO

A diferencia de TF (que solo cuenta ocurrencias), **TF-IDF** pondera cada término según qué tan frecuente es dentro de un tweet (TF) pero penalizando los términos que aparecen en muchos tweets distintos (IDF). Esto suele darle más peso a palabras discriminativas y menos a palabras muy comunes.

Reutilizamos los mismos `X_train`/`y_train`, `X_dev`/`y_dev` y `X_test`/`y_test` de la sección anterior, y repetimos el mismo procedimiento (elegir `C` con el split de validación y evaluar sobre test) pero con `TfidfVectorizer` en lugar de `CountVectorizer`.

In [ ]:
# Pipeline con TF-IDF y LASSO
pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

val_results = []

In [ ]:
%%time
for C in C_values:
    pipeline_tfidf.set_params(classifier__C=C)

    pipeline_tfidf.fit(X_train, y_train)
    dev_proba = pipeline_tfidf.predict_proba(X_dev)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })

val_results_df = pd.DataFrame(val_results)

best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_tfidf = val_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C_tfidf:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tfidf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

final_pipeline_tfidf.fit(X_train, y_train)
y_pred = final_pipeline_tfidf.predict(X_test)
y_pred_proba = final_pipeline_tfidf.predict_proba(X_test)[:, 1]

results_tfidf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

for metric, value in results_tfidf.items():
    print(f"{metric}: {value:.3f}")

## Interpretación de los coeficientes

Repetimos el mismo análisis que en la sección de TF, pero con una diferencia importante en cómo se leen los coeficientes acá.

- **LASSO sigue seleccionando variables**: de nuevo, la gran mayoría de los coeficientes del vocabulario quedan en cero; solo sobrevive un subconjunto chico.
- **El odds ratio ya no tiene la misma lectura "por ocurrencia".** En TF cada feature era un conteo entero (0, 1, 2, ...), así que `exp(beta)` se interpretaba como "cuánto multiplica las odds cada aparición adicional". Acá cada feature es un **score TF-IDF continuo**: pondera la frecuencia del término en el tweet por qué tan raro es en todo el corpus (IDF), y además cada vector de tweet se normaliza (norma L2). Un "incremento de una unidad" en ese score no corresponde a nada tan intuitivo como "una palabra más" — por eso acá **no** mostramos la columna de odds ratio. El signo (empuja hacia `HS=1` o `HS=0`) y el orden relativo entre términos siguen siendo perfectamente interpretables; lo que se pierde es la lectura "por ocurrencia" en unidades naturales.

In [ ]:
feature_names_tfidf = final_pipeline_tfidf.named_steps['vectorizer'].get_feature_names_out()
coefs_tfidf = final_pipeline_tfidf.named_steps['classifier'].coef_[0]

coef_df_tfidf = pd.DataFrame({'feature': feature_names_tfidf, 'coef': coefs_tfidf})

n_total = len(coef_df_tfidf)
n_nonzero = int((coef_df_tfidf['coef'] != 0).sum())
print(f"Features totales: {n_total} | con coeficiente != 0 (seleccionadas por LASSO): "
      f"{n_nonzero} ({n_nonzero / n_total * 100:.1f}%)")

top_pos_tfidf = coef_df_tfidf.sort_values('coef', ascending=False).head(15)
top_neg_tfidf = coef_df_tfidf.sort_values('coef', ascending=True).head(15)

print("\nTop 15 términos que más empujan hacia HS=1 (discurso de odio):")
display(top_pos_tfidf)

print("\nTop 15 términos que más empujan hacia HS=0 (no odio):")
display(top_neg_tfidf)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_terms_tfidf = pd.concat([top_neg_tfidf, top_pos_tfidf]).sort_values('coef')
colors_coef = ['#e34948' if c > 0 else '#2a78d6' for c in top_terms_tfidf['coef']]

fig, ax = plt.subplots(figsize=(7, 8))
fig.patch.set_facecolor('#fcfcfb')
sns.barplot(data=top_terms_tfidf, x='coef', y='feature', hue='feature',
            palette=colors_coef, legend=False, ax=ax)
ax.axvline(0, color='#898781', linewidth=1)
ax.set_facecolor('#fcfcfb')
ax.set_xlabel('coeficiente (beta)', color='#52514e')
ax.set_ylabel('')
ax.set_title('Términos más predictivos — TF-IDF + LASSO', color='#0b0b0b', fontsize=11)
ax.tick_params(colors='#52514e')
sns.despine(ax=ax, left=True)
fig.tight_layout()
plt.show()

# Word embeddings como features

## Idea general

En lugar de representar cada tweet como un vector disperso de conteos (TF) o pesos (TF-IDF) sobre el vocabulario, ahora vamos a representarlo como el **promedio de los vectores de embedding preentrenados** de sus palabras. Usamos los embeddings estáticos en español **SBWCE** (`SBW-vectors-300-min5`, ya descargados al inicio del notebook), donde cada palabra se representa con un vector denso de 300 dimensiones.

El preprocesamiento acá es más simple: solo eliminamos URLs y menciones, y reemplazamos números por `DIGITO`. No hace falta sacar puntuación ni acentos porque las palabras que no estén en el vocabulario de los embeddings simplemente se ignoran (*out-of-vocabulary*).

Como en las secciones anteriores, vectorizamos por separado los tweets en español de `train`, `dev` y `test`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
from gensim.models import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Descargas necesarias para tokenización
nltk.download('punkt_tab')
nltk.download('punkt')

# Preprocesamiento simple: solo removemos URLs, menciones y reemplazamos dígitos
def preprocess_text_embed(text):
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\d+', 'DIGITO', text)
    return text

# Cargamos un split y nos quedamos con los tweets en español
def load_hateval_split_embed(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    df['text_clean'] = df['text'].apply(preprocess_text_embed)
    return df

train_df = load_hateval_split_embed(train_path)
dev_df = load_hateval_split_embed(dev_path)
test_df = load_hateval_split_embed(test_path)

# Cargamos el modelo de word embeddings
def load_embeddings(path):
    print("Cargando embeddings...")
    return KeyedVectors.load_word2vec_format(path, binary=True)

word_vectors = load_embeddings("./models/SBW-vectors-300-min5.bin")

def get_mean_vector(text, word_vectors, vector_size=300):
    """
    Obtiene el embedding promedio de un texto: lo tokeniza, busca el vector
    preentrenado de cada palabra y devuelve el promedio de esos vectores
    como un único vector de `vector_size` dimensiones. Las palabras que no
    están en el vocabulario de los embeddings (out-of-vocabulary) se
    ignoran. Es una forma simple -pero con pérdida- de resumir un texto de
    largo variable en un vector de tamaño fijo: mezcla el significado de
    todas las palabras en un solo punto del espacio semántico, sin
    conservar el orden ni la estructura de la oración.

    Usamos `KeyedVectors.get_mean_vector`, que ya hace el promedio (y el
    manejo de palabras out-of-vocabulary) internamente. `pre_normalize=False`
    para promediar los vectores tal cual, sin normalizarlos primero a
    norma 1 (que es el comportamiento por default de gensim).
    """
    words = word_tokenize(text.lower())
    if not words:
        return np.zeros(vector_size)
    return word_vectors.get_mean_vector(words, pre_normalize=False, ignore_missing=True)

# Convertimos los tweets de un split a vectores
def vectorize_split(df, word_vectors):
    print("Vectorizando tweets...")
    vectors = [get_mean_vector(text, word_vectors) for text in df['text_clean']]
    X = pd.DataFrame(vectors, columns=[f'V{i+1}' for i in range(300)])
    X['id'] = df['id'].values
    y = df['HS'].values
    return X, y

X_train_embed_full, y_train = vectorize_split(train_df, word_vectors)
X_dev_embed_full, y_dev = vectorize_split(dev_df, word_vectors)
X_test_embed_full, y_test = vectorize_split(test_df, word_vectors)

# Guardamos los ids por separado y nos quedamos solo con las columnas de vectores
train_ids = X_train_embed_full['id']
dev_ids = X_dev_embed_full['id']
test_ids = X_test_embed_full['id']
X_train_embed = X_train_embed_full.drop('id', axis=1)
X_dev_embed = X_dev_embed_full.drop('id', axis=1)
X_test_embed = X_test_embed_full.drop('id', axis=1)

In [ ]:
X_train_embed.head()

## Búsqueda de hiperparámetros usando el split de validación

Repetimos el mismo esquema que en las secciones anteriores: para cada valor de `C` entrenamos sobre `X_train_embed`/`y_train` y evaluamos el ROC AUC sobre `X_dev_embed`/`y_dev`, y nos quedamos con el que mejor performa en `dev`.

In [ ]:
# Grilla de valores para C
C_values = np.logspace(-10, 1, 30)

val_results = []

In [ ]:
%%time
print("Buscando el mejor valor de C...")
for C in C_values:
    model = LogisticRegression(
        C=C,
        penalty='l1',
        solver='liblinear',
        random_state=234
    )

    model.fit(X_train_embed, y_train)
    dev_proba = model.predict_proba(X_dev_embed)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })

val_results_df = pd.DataFrame(val_results)

best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_embed = val_results_df.loc[best_idx, 'C']

print(f"\nMejor valor de C: {best_C_embed:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_model_embed = LogisticRegression(
    C=best_C_embed,
    penalty='l1',
    solver='liblinear',
    random_state=234
)

final_model_embed.fit(X_train_embed, y_train)
y_pred = final_model_embed.predict(X_test_embed)
y_pred_proba = final_model_embed.predict_proba(X_test_embed)[:, 1]

results_embed = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("Resultados finales:")
for metric, value in results_embed.items():
    print(f"{metric}: {value:.3f}")

## Los coeficientes acá no son interpretables

En TF y TF-IDF pudimos leer directamente `coef_` porque cada columna del vectorizador **es** una palabra: el coeficiente de la columna "zorra" nos dice literalmente cuánto pesa la palabra "zorra". Acá eso se rompe.

`final_model_embed.coef_` también tiene 300 valores (uno por columna de `X_train_embed`), pero esas 300 columnas son `V1`, `V2`, ..., `V300`: **dimensiones abstractas** del espacio vectorial que aprendió SBWCE al entrenarse sobre un corpus enorme, no conceptos con un nombre. No existe un `get_feature_names_out()` para embeddings porque no hay "nombres" que devolver — cada dimensión mezcla, de una forma que no fue diseñada para ser legible por humanos, información sobre miles de palabras distintas.

Así que aunque técnicamente podamos calcular "el coeficiente de V142 es 0.6", esa afirmación no nos dice nada sobre *qué tipo de tweet* empuja el modelo hacia `HS=1` — a diferencia de "el coeficiente de 'zorra' es 1.5" en TF, que sí es una afirmación con contenido. Es el precio de usar una representación densa y aprendida en otro corpus: se gana la capacidad de capturar relaciones semánticas entre palabras (sinónimos, analogías), pero se pierde la interpretabilidad directa palabra-por-palabra que tienen TF y TF-IDF por construcción.

(Para interpretar qué "significa" una dimensión de embeddings hacen falta técnicas indirectas — por ejemplo, buscar qué palabras del vocabulario tienen los valores más altos/bajos en esa dimensión particular, o proyectar el espacio a 2D con t-SNE/UMAP — que quedan fuera del alcance de este notebook.)

In [ ]:
# A diferencia de TF/TF-IDF, coef_ tiene 300 valores pero sin un vocabulario
# que los traduzca a algo legible: son solo índices de columna
coefs_embed = final_model_embed.coef_[0]
coef_df_embed = pd.DataFrame({
    'dimension': X_train_embed.columns,
    'coef': coefs_embed
})

n_total = len(coef_df_embed)
n_nonzero = int((coef_df_embed['coef'] != 0).sum())
print(f"Features totales: {n_total} | con coeficiente != 0 (seleccionadas por LASSO): "
      f"{n_nonzero} ({n_nonzero / n_total * 100:.1f}%)")

print("\nTop 5 dimensiones con mayor coeficiente (compará esto con las tablas de TF/TF-IDF):")
display(coef_df_embed.sort_values('coef', ascending=False).head())

# Comparación de resultados

Juntamos las métricas de desempeño de las tres representaciones, evaluadas sobre `test`.

En TF y TF-IDF pudimos interpretar directamente los coeficientes del modelo (qué palabras empujan hacia `HS=1` o `HS=0`). En embeddings vimos que esa interpretación directa no es posible: la representación es más "opaca" por construcción, aunque eso no significa necesariamente que prediga mejor o peor — lo vemos en la tabla de abajo.

In [ ]:
results_tf_df = pd.DataFrame([results_tf], index=['TF'])
results_tfidf_df = pd.DataFrame([results_tfidf], index=['TF-IDF'])
results_embed_df = pd.DataFrame([results_embed], index=['Word Embeddings'])

comparison_df = pd.concat([results_tf_df, results_tfidf_df, results_embed_df])

print("Comparación de resultados de los modelos (evaluados sobre test):")
display(comparison_df)

# Conclusiones

Estos son los resultados de una corrida real del notebook (con el SBWCE real, no simulado):

| | ROC AUC | F1 | Precision | Recall |
|---|---|---|---|---|
| TF | 0.777 | 0.665 | 0.636 | 0.697 |
| TF-IDF | 0.773 | 0.665 | 0.591 | 0.759 |
| Word Embeddings | 0.759 | 0.596 | 0.674 | 0.533 |

(Los valores exactos van a variar entre corridas — lo que importa es el patrón, no el número puntual.)

## Desempeño: TF y TF-IDF le ganan a embeddings acá

TF y TF-IDF empatan en F1 (0.665), unos 7 puntos por encima de embeddings (0.596). Tiene sentido para *hate speech*: la señal discriminativa suele estar en palabras o frases puntuales (insultos, *slurs* específicos), y una LASSO sobre 1-2 gramas puede aislar exactamente esos términos con un coeficiente alto. El promedio de embeddings, en cambio, diluye esa señal — mezcla todas las palabras del tweet en un solo vector de 300 dimensiones, perdiendo cuál palabra puntual disparó el odio. Con solo 4500 tweets de entrenamiento tampoco hay tanto margen para que embeddings compense por generalización semántica.

**El trade-off precision/recall es la diferencia más interesante y práctica.** TF-IDF prioriza recall (0.76) a costa de precision (0.59): atrapa más discurso de odio real pero con más falsos positivos. Embeddings hace lo opuesto: alta precision (0.67), recall bajo (0.53) — es conservador, se pierde casi la mitad del odio real. Si el costo de dejar pasar contenido de odio es alto, TF-IDF es preferible; si el costo de un falso positivo es alto (por ejemplo, penalizar cuentas por error), embeddings.

## Interpretabilidad: otro punto a favor de TF/TF-IDF

Además de predecir mejor acá, TF y TF-IDF tienen una ventaja que no aparece en las métricas: son **interpretables por construcción**. Pudimos mostrar exactamente qué palabras y bigramas asocia el modelo con `HS=1` y con `HS=0`, con una lectura directa (para TF, incluso cuantificable como odds ratio). Con embeddings esa lectura no existe: los 300 coeficientes del modelo final corresponden a dimensiones abstractas del espacio vectorial preentrenado, sin un vocabulario que las traduzca a algo legible.

Esto importa más allá de la curiosidad académica: si en algún momento hay que auditar por qué el modelo marcó un tweet como discurso de odio (por ejemplo, para explicarle una decisión a un usuario, o para detectar sesgos del modelo), TF y TF-IDF permiten esa auditoría de forma directa; embeddings no.

## Conclusión práctica

Para esta tarea puntual, con este modelo lineal y esta cantidad de datos, **TF-IDF es la opción dominante**: mismo F1 que TF, mejor recall, y — como TF — permite interpretar directamente qué aprendió el modelo. Embeddings pierde en desempeño y además sacrifica la interpretabilidad, sin compensarlo con una ventaja clara acá — un resultado legítimo y pedagógicamente útil: una representación "más sofisticada" no gana automáticamente, sobre todo cuando se la usa de forma simple (promedio de vectores) en una tarea donde palabras puntuales importan más que el significado semántico agregado, y con pocos datos de entrenamiento.

# Ejercicio

Entrenar y tunear un Random Forest usando features construidas con TF-IDF y otro con embeddings, para clasificar los tweets en español del dataset HatEval. Usá el mismo esquema que en este notebook: elegí los hiperparámetros con el split de `dev` y reportá el desempeño final sobre `test` (ROC AUC, accuracy, precision, recall, F1).

¿Cuál resulta más eficaz? ¿Por qué?

Como desafío adicional: ¿qué esperarías que pase si usás estos mismos embeddings en español (SBWCE) para vectorizar los tweets en **inglés** del dataset (`language == 'en'`)?

In [ ]:
###